# Détection des maladies pulmonaires par CNN : expérimentation

**Mémoire de Master, ALAWO Adeshina Néhémiah (ESGIS)**

Ce notebook entraîne et évalue DenseNet-121 (transfer learning en deux phases) selon le protocole du Chapitre 2,
puis produit tout ce qu'attend le Chapitre 3 : Tableau 3.1, Figures 3.1 à 3.4, et un récapitulatif chiffré.

| Tâche | Données | Rôle dans le mémoire |
|---|---|---|
| `pneumonie` | Kermany et al. (2018), 5 856 radiographies pédiatriques | Ligne « Pneumonie » du Tableau 3.1 |
| `tuberculose` | Montgomery + Shenzhen (Jaeger et al., 2014), 800 radiographies | Ligne « Tuberculose » |
| `bacterien_viral` | Kermany, images de pneumonie uniquement | Sous-tâche bactérienne vs virale (§ 3.2) |

## Mode d'emploi

1. **GPU** : menu *Exécution → Modifier le type d'exécution → GPU T4*.
2. **Clé Kaggle** (téléchargement des données) : sur kaggle.com → *Settings → API → Create New Token*.
   Dans Colab, panneau de gauche 🔑 *Secrets* : ajouter `KAGGLE_USERNAME` et `KAGGLE_KEY`
   (ou `KAGGLE_API_TOKEN` pour les jetons récents) et cocher « Accès au notebook ».
3. *Exécution → Tout exécuter*. Autoriser l'accès à Google Drive quand Colab le demande.
4. Durée indicative : 1 h à 1 h 30 sur T4. Les résultats sont enregistrés au fur et à mesure dans
   `Mon Drive/memoire_resultats/`. **Si Colab se déconnecte, relancer « Tout exécuter »** :
   les tâches déjà terminées sont sautées.
5. À la fin, un fichier `memoire_resultats.zip` est téléchargé : c'est lui qui sert à remplir le Chapitre 3.

In [ ]:
%pip install -q kagglehub tabulate

## 1. Configuration
Les valeurs ci-dessous sont celles du Chapitre 2. Toute modification doit y être reportée.

In [ ]:
import os, re, io, copy, json, time, random, shutil, hashlib, platform, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from PIL import Image
import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

CONFIG = {
    'seed': 42,
    'taille_cache': 256,           # images stockées en 256 × 256 (prétraitement unique)
    'taille_entree': 224,          # entrée native de DenseNet-121
    'batch_size': 32,
    'num_workers': 2,
    'partition': (0.70, 0.15, 0.15),
    'dropout': 0.5,
    'weight_decay': 1e-5,          # régularisation L2 dans Adam
    'plateau': {'factor': 0.5, 'patience': 5},   # ReduceLROnPlateau
    # Phase 1, feature extraction : backbone gelé, seule la tête apprend
    'phase1': {'lr': 1e-4, 'max_epochs': 10, 'patience': 10},
    # Phase 2, fine-tuning : dégel progressif des blocs profonds, taux d'apprentissage réduit
    'phase2': {'lr': 1e-5, 'max_epochs': 40, 'patience': 10,
               'degel': [(0, ['denseblock4', 'norm5']),           # dès la 1re époque de phase 2
                         (5, ['transition3', 'denseblock3'])]},   # à partir de la 6e époque
    'poids_imagenet': True,
    'bootstrap': 1000,             # tirages pour les intervalles de confiance à 95 %
}
TACHES_A_LANCER = ['pneumonie', 'tuberculose', 'bacterien_viral']
REENTRAINER = False                # True : refait les tâches même si elles sont déjà sur le Drive


def fixer_graine(seed=CONFIG['seed']):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fixer_graine()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'
if DEVICE.type != 'cuda':
    print("⚠️  Aucun GPU détecté : Exécution → Modifier le type d'exécution → GPU T4, puis relancer.")

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/memoire_resultats')
    CACHE_DIR = Path('/content/cache')
else:
    OUT_DIR = Path('memoire_resultats')
    CACHE_DIR = Path('cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
print(f'Matériel : {GPU} | PyTorch {torch.__version__} | torchvision {torchvision.__version__}')
print(f'Résultats : {OUT_DIR.resolve()}')

## 2. Téléchargement des données (Kaggle)

In [ ]:
try:
    from google.colab import userdata
    for cle in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_API_TOKEN'):
        try:
            os.environ[cle] = userdata.get(cle)
        except Exception:
            pass
except ImportError:
    pass

import kagglehub
try:
    KERMANY_DIR = Path(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia'))
    TB_DIR = Path(kagglehub.dataset_download('kmader/pulmonary-chest-xray-abnormalities'))
except Exception as e:
    raise RuntimeError("Téléchargement Kaggle impossible : vérifier les secrets KAGGLE_* "
                       "(voir le mode d'emploi en haut du notebook).") from e
print(KERMANY_DIR, TB_DIR, sep='\n')

## 3. Inventaire des images et identification des patients

Le partitionnement se fait **par patient** : toutes les radiographies d'un même patient vont dans le même
ensemble. L'identifiant patient est lu dans le nom de fichier (`personN` pour Kermany-pneumonie,
`IM-NNNN` pour Kermany-normal, numéro de dossier pour Montgomery et Shenzhen).
Les répartitions officielles de Kermany (dont un ensemble de validation de 16 images) ne sont pas reprises :
toutes les images sont regroupées puis repartagées 70 / 15 / 15.

In [ ]:
EXTENSIONS = {'.jpeg', '.jpg', '.png'}


def indexer_kermany(racine):
    lignes, vues = [], set()
    for p in sorted(Path(racine).rglob('*')):
        if p.suffix.lower() not in EXTENSIONS or '__MACOSX' in p.parts:
            continue
        classe = p.parent.name.upper()
        if classe not in ('NORMAL', 'PNEUMONIA'):
            continue
        # L'archive Kaggle contient chest_xray/ en double : on dédoublonne sur (dossier, classe, nom)
        cle = (p.parent.parent.name.lower(), classe, p.name)
        if cle in vues:
            continue
        vues.add(cle)
        if classe == 'NORMAL':
            m = re.search(r'IM-(\d+)', p.name)
            patient, sous_type = f"N-{m.group(1) if m else p.stem}", 'normal'
        else:
            m = re.match(r'(person\d+)_(bacteria|virus)', p.name, re.I)
            patient = f"P-{m.group(1).lower() if m else p.stem}"
            sous_type = m.group(2).lower() if m else 'inconnu'
        lignes.append({'path': str(p), 'source': 'Kermany', 'patient': patient,
                       'classe': classe, 'sous_type': sous_type})
    return pd.DataFrame(lignes)


def indexer_tuberculose(racine):
    lignes, vues = [], set()
    for p in sorted(Path(racine).rglob('*.png')):
        m = re.match(r'^(MCUCXR|CHNCXR)_(\d+)_([01])\.png$', p.name, re.I)
        if not m or 'mask' in str(p.parent).lower() or p.name.upper() in vues:
            continue
        vues.add(p.name.upper())
        lignes.append({'path': str(p),
                       'source': 'Montgomery' if m.group(1).upper() == 'MCUCXR' else 'Shenzhen',
                       'patient': f'{m.group(1).upper()}_{m.group(2)}',
                       'classe': 'TB' if m.group(3) == '1' else 'NORMAL', 'sous_type': ''})
    return pd.DataFrame(lignes)


kermany = indexer_kermany(KERMANY_DIR)
tb = indexer_tuberculose(TB_DIR)
print(f'Kermany : {len(kermany)} images (attendu : 5 856)')
print(kermany.groupby(['classe', 'sous_type']).size().to_string(), '\n')
print(f'Montgomery + Shenzhen : {len(tb)} images (attendu : 800)')
print(tb.groupby(['source', 'classe']).size().to_string())

TACHES = {
    'pneumonie': {
        'titre': 'Pneumonie', 'libelles': ('Normal', 'Pneumonie'), 'pathologie': True,
        'df': kermany.assign(y=(kermany.classe == 'PNEUMONIA').astype(int)),
    },
    'tuberculose': {
        'titre': 'Tuberculose', 'libelles': ('Normal', 'Tuberculose'), 'pathologie': True,
        'df': tb.assign(y=(tb.classe == 'TB').astype(int)),
    },
    'bacterien_viral': {
        'titre': 'Pneumonie bactérienne vs virale', 'libelles': ('Bactérienne', 'Virale'), 'pathologie': False,
        'df': kermany[kermany.sous_type.isin(['bacteria', 'virus'])]
              .assign(y=lambda d: (d.sous_type == 'virus').astype(int)),
    },
}

## 4. Partitionnement 70 / 15 / 15 au niveau patient, stratifié par classe

In [ ]:
def partitionner(df, seed=CONFIG['seed']):
    df = df.reset_index(drop=True).copy()
    _, p_val, p_test = CONFIG['partition']
    # Étiquette du patient = classe majoritaire de ses images (sert uniquement à stratifier)
    patients = df.groupby('patient')['y'].agg(lambda s: int(s.mean() >= 0.5)).reset_index()
    reste, test = train_test_split(patients, test_size=p_test, stratify=patients.y, random_state=seed)
    train, val = train_test_split(reste, test_size=p_val / (1 - p_test), stratify=reste.y, random_state=seed)
    ensemble = {**dict.fromkeys(train.patient, 'train'), **dict.fromkeys(val.patient, 'val'),
                **dict.fromkeys(test.patient, 'test')}
    df['split'] = df.patient.map(ensemble)
    assert df.groupby('patient').split.nunique().max() == 1, 'Un patient est dans deux ensembles'
    return df


DONNEES, lignes_resume = {}, []
for cle, t in TACHES.items():
    d = partitionner(t['df'])
    DONNEES[cle] = d
    for split in ('train', 'val', 'test'):
        s = d[d.split == split]
        lignes_resume.append({'tâche': t['titre'], 'ensemble': split, 'images': len(s),
                              'patients': s.patient.nunique(),
                              'négatifs': int((s.y == 0).sum()), 'positifs': int((s.y == 1).sum()),
                              'classes (nég. / pos.)': ' / '.join(t['libelles'])})
REPARTITION = pd.DataFrame(lignes_resume)
REPARTITION.to_csv(OUT_DIR / 'repartition_donnees.csv', index=False)
REPARTITION

## 5. Prétraitement unique : niveaux de gris, 256 × 256, cache local
Les radiographies Montgomery sont en 16 bits : elles sont ramenées sur 8 bits par normalisation min-max.

In [ ]:
def charger_gris(chemin):
    img = Image.open(chemin)
    if img.mode in ('I;16', 'I;16B', 'I;16L', 'I', 'F'):
        a = np.asarray(img, dtype=np.float32)
        a = (a - a.min()) / max(float(a.max() - a.min()), 1e-6) * 255
        img = Image.fromarray(a.astype(np.uint8))
    return img.convert('L')


def mettre_en_cache(chemin):
    sortie = CACHE_DIR / (hashlib.md5(chemin.encode()).hexdigest() + '.png')
    if not sortie.exists():
        c = CONFIG['taille_cache']
        charger_gris(chemin).resize((c, c), Image.LANCZOS).save(sortie)
    return str(sortie)


tous = sorted(set().union(*(set(d.path) for d in DONNEES.values())))
t0 = time.time()
with ThreadPoolExecutor(8) as pool:
    correspondance = dict(zip(tous, pool.map(mettre_en_cache, tous)))
for d in DONNEES.values():
    d['cache'] = d.path.map(correspondance)
print(f'{len(tous)} images prétraitées en {time.time() - t0:.0f} s')

## 6. Jeux de données, augmentation, modèle

In [ ]:
NORMALISATION = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # statistiques ImageNet
TAILLE = CONFIG['taille_entree']

# Augmentation réservée à l'entraînement (Chapitre 2) : rotation ±15°, zoom ±10 %, rognage aléatoire,
# retournement horizontal, luminosité/contraste. Pas de retournement vertical ni de déformation élastique.
TF_ENTRAINEMENT = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),          # pseudo-RGB pour les poids ImageNet
    transforms.RandomAffine(degrees=15, scale=(0.9, 1.1)),
    transforms.RandomResizedCrop(TAILLE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), NORMALISATION,
])
TF_EVALUATION = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((TAILLE, TAILLE)),
    transforms.ToTensor(), NORMALISATION,
])


class Radiographies(Dataset):
    def __init__(self, df, tf):
        self.chemins, self.y, self.tf = df.cache.tolist(), df.y.astype('float32').tolist(), tf

    def __len__(self):
        return len(self.chemins)

    def __getitem__(self, i):
        return self.tf(Image.open(self.chemins[i]).convert('L')), torch.tensor(self.y[i])


def init_worker(worker_id):
    graine = torch.initial_seed() % 2**32
    np.random.seed(graine)
    random.seed(graine)


def chargeurs(df):
    g = torch.Generator().manual_seed(CONFIG['seed'])
    commun = {'batch_size': CONFIG['batch_size'], 'num_workers': CONFIG['num_workers'],
              'pin_memory': DEVICE.type == 'cuda', 'worker_init_fn': init_worker}
    train = df[df.split == 'train']
    return {
        'train': DataLoader(Radiographies(train, TF_ENTRAINEMENT), shuffle=True, generator=g,
                            drop_last=len(train) > CONFIG['batch_size'], **commun),
        'val': DataLoader(Radiographies(df[df.split == 'val'], TF_EVALUATION), shuffle=False, **commun),
        'test': DataLoader(Radiographies(df[df.split == 'test'], TF_EVALUATION), shuffle=False, **commun),
    }


class DenseNetRadio(nn.Module):
    """DenseNet-121 dont la tête ImageNet est remplacée par GAP → Dropout(0,5) → couche dense à 1 sortie (sigmoïde)."""

    def __init__(self, poids_imagenet=True, dropout=0.5):
        super().__init__()
        poids = torchvision.models.DenseNet121_Weights.IMAGENET1K_V1 if poids_imagenet else None
        base = torchvision.models.densenet121(weights=poids)
        self.features = base.features
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(base.classifier.in_features, 1)
        self.activations = self.gradients = None

    def forward(self, x, garder_activations=False):
        a = F.relu(self.features(x))
        if garder_activations:                       # pour Grad-CAM : dernière carte de caractéristiques 7 × 7
            self.activations = a
            a.register_hook(lambda g: setattr(self, 'gradients', g))
        z = F.adaptive_avg_pool2d(a, 1).flatten(1)
        return self.fc(self.dropout(z)).squeeze(1)


def degeler(model, blocs):
    for nom, module in model.features.named_children():
        if nom in blocs:
            for p in module.parameters():
                p.requires_grad = True


def geler_backbone(model):
    for p in model.features.parameters():
        p.requires_grad = False


def mode_entrainement(model):
    # Les blocs gelés restent en mode évaluation : leurs statistiques BatchNorm ImageNet ne bougent pas
    model.train()
    for module in model.features.children():
        if not any(p.requires_grad for p in module.parameters()):
            module.eval()

## 7. Entraînement en deux phases, évaluation, Grad-CAM

In [ ]:
def une_epoque(model, loader, critere, optimiseur=None, scaler=None):
    entrainement = optimiseur is not None
    mode_entrainement(model) if entrainement else model.eval()
    perte, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(entrainement):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
                logits = model(x)
            loss = critere(logits.float(), y)
            if entrainement:
                optimiseur.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimiseur)
                scaler.update()
            perte += loss.item() * len(y)
            correct += ((logits > 0).float() == y).sum().item()
            n += len(y)
    return perte / n, correct / n


def entrainer_phase(model, loaders, critere, nom_phase, cfg, historique):
    optimiseur = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=CONFIG['weight_decay'])
    planif = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiseur, mode='min', **CONFIG['plateau'])
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == 'cuda')
    meilleure, attente, meilleurs_poids, epoque_retenue = float('inf'), 0, None, None
    for ep in range(cfg['max_epochs']):
        for debut, blocs in cfg.get('degel', []):
            if ep == debut:
                degeler(model, blocs)
                print(f'   dégel : {", ".join(blocs)}')
        t0 = time.time()
        loss, acc = une_epoque(model, loaders['train'], critere, optimiseur, scaler)
        val_loss, val_acc = une_epoque(model, loaders['val'], critere)
        planif.step(val_loss)
        historique.append({'phase': nom_phase, 'epoque': len(historique) + 1, 'loss': loss, 'acc': acc,
                           'val_loss': val_loss, 'val_acc': val_acc,
                           'lr': optimiseur.param_groups[0]['lr'], 'duree_s': time.time() - t0})
        print(f"{nom_phase} | époque {len(historique):>2} | perte {loss:.4f} / val {val_loss:.4f} "
              f"| exactitude {acc:.3f} / val {val_acc:.3f} | {time.time() - t0:.0f} s")
        if val_loss < meilleure - 1e-4:
            meilleure, attente, epoque_retenue = val_loss, 0, len(historique)
            meilleurs_poids = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            attente += 1
            if attente >= cfg['patience']:
                print(f'   arrêt anticipé (pas d\'amélioration depuis {attente} époques)')
                break
    model.load_state_dict(meilleurs_poids)
    return epoque_retenue


@torch.no_grad()
def predire(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            ps.append(torch.sigmoid(model(x.to(DEVICE)).float()).cpu())
        ys.append(y)
    return torch.cat(ys).numpy().astype(int), torch.cat(ps).numpy()


def metriques(y, p, seuil):
    pred = (p >= seuil).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    rappel = tp / (tp + fn) if tp + fn else 0.0
    return {'precision': precision, 'rappel': rappel,
            'specificite': tn / (tn + fp) if tn + fp else 0.0,
            'f1': 2 * precision * rappel / (precision + rappel) if precision + rappel else 0.0,
            'exactitude': (tp + tn) / len(y),
            'auc': roc_auc_score(y, p) if len(np.unique(y)) == 2 else float('nan'),
            'vp': int(tp), 'fp': int(fp), 'fn': int(fn), 'vn': int(tn)}


def seuil_youden(y, p):
    """Seuil maximisant sensibilité + spécificité − 1, choisi sur la VALIDATION (jamais sur le test)."""
    fpr, tpr, seuils = roc_curve(y, p)
    return float(min(seuils[np.argmax(tpr - fpr)], 1.0))


def intervalles_bootstrap(y, p, seuil, n=CONFIG['bootstrap']):
    rng = np.random.default_rng(CONFIG['seed'])
    tirages = {k: [] for k in ('precision', 'rappel', 'specificite', 'f1', 'auc')}
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        m = metriques(y[idx], p[idx], seuil)
        for k in tirages:
            tirages[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in tirages.items()}


def gradcam(model, image_pil):
    model.eval()
    x = TF_EVALUATION(image_pil).unsqueeze(0).to(DEVICE).requires_grad_(True)
    model.zero_grad(set_to_none=True)
    logit = model(x, garder_activations=True)
    logit.sum().backward()                                   # gradient du score « pathologique »
    a, g = model.activations.detach()[0], model.gradients.detach()[0]
    carte = F.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0))
    carte = carte / (carte.max() + 1e-8)
    carte = F.interpolate(carte[None, None], size=(TAILLE, TAILLE), mode='bilinear', align_corners=False)[0, 0]
    return carte.cpu().numpy(), float(torch.sigmoid(logit).item())


def superposer(image_pil, carte, alpha=0.4):
    fond = np.asarray(image_pil.convert('L').resize((TAILLE, TAILLE)), dtype=np.float32)[..., None] / 255
    chaleur = matplotlib.colormaps['jet'](carte)[..., :3]
    return np.clip((1 - alpha) * fond + alpha * chaleur, 0, 1)


def sauver_gradcam(model, test, dossier):
    """Un VP représentatif (score médian parmi les VP) et le FN le plus franc, plus une galerie pour choisir."""
    test = test.reset_index(drop=True)
    vp = test[(test.y == 1) & (test.pred == 1)].sort_values('proba')
    fn = test[(test.y == 1) & (test.pred == 0)].sort_values('proba')
    choix = {}
    if len(vp):
        choix['VP'] = vp.iloc[len(vp) // 2]
    if len(fn):
        choix['FN'] = fn.iloc[0]
    galerie = list(vp.iloc[np.linspace(0, len(vp) - 1, min(len(vp), 6)).astype(int)].itertuples()) if len(vp) else []
    galerie += list(fn.head(6).itertuples())
    (dossier / 'gradcam_galerie').mkdir(exist_ok=True)
    infos = {}
    for type_cas, ligne in choix.items():
        img = Image.open(ligne.cache)
        carte, proba = gradcam(model, img)
        plt.imsave(dossier / f'gradcam_{type_cas}.png', superposer(img, carte))
        infos[type_cas] = {'image': ligne.path, 'patient': ligne.patient, 'proba': proba}
    for r in galerie:
        img = Image.open(r.cache)
        carte, _ = gradcam(model, img)
        type_cas = 'VP' if r.pred == 1 else 'FN'
        plt.imsave(dossier / 'gradcam_galerie' / f'{type_cas}_{r.proba:.3f}_{r.patient}.png', superposer(img, carte))
    return infos

## 8. Lancement des expériences
Chaque tâche enregistre sur le Drive : modèle, historique, prédictions du test, métriques et cartes Grad-CAM.

In [ ]:
RESULTATS = {}
for cle in TACHES_A_LANCER:
    dossier = OUT_DIR / cle
    dossier.mkdir(exist_ok=True)
    if (dossier / 'metriques.json').exists() and not REENTRAINER:
        RESULTATS[cle] = json.loads((dossier / 'metriques.json').read_text(encoding='utf-8'))
        print(f'✔ {cle} déjà terminé, sauté (REENTRAINER = True pour le refaire)')
        continue

    print(f'\n===== {TACHES[cle]["titre"]} =====')
    df = DONNEES[cle]
    fixer_graine()
    loaders = chargeurs(df)
    model = DenseNetRadio(CONFIG['poids_imagenet'], CONFIG['dropout']).to(DEVICE)

    # Perte pondérée : le poids des positifs compense le déséquilibre des classes de l'entraînement
    train = df[df.split == 'train']
    poids_positif = (train.y == 0).sum() / max((train.y == 1).sum(), 1)
    critere = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(poids_positif), device=DEVICE))

    historique, debut = [], time.time()
    geler_backbone(model)
    entrainer_phase(model, loaders, critere, 'Phase 1', CONFIG['phase1'], historique)
    fin_phase1 = len(historique)
    epoque_retenue = entrainer_phase(model, loaders, critere, 'Phase 2', CONFIG['phase2'], historique)
    duree = time.time() - debut
    torch.save(model.state_dict(), dossier / 'modele.pt')
    pd.DataFrame(historique).to_csv(dossier / 'historique.csv', index=False)

    y_val, p_val = predire(model, loaders['val'])
    seuil = seuil_youden(y_val, p_val)
    y_test, p_test = predire(model, loaders['test'])
    test = df[df.split == 'test'].reset_index(drop=True).assign(proba=p_test, pred=(p_test >= seuil).astype(int))
    test[['path', 'source', 'patient', 'y', 'proba', 'pred']].to_csv(dossier / 'predictions_test.csv', index=False)

    RESULTATS[cle] = {
        'titre': TACHES[cle]['titre'], 'libelles': TACHES[cle]['libelles'],
        'pathologie': TACHES[cle]['pathologie'],
        'seuil_validation': seuil,
        'test': metriques(y_test, p_test, seuil),
        'test_seuil_0_5': metriques(y_test, p_test, 0.5),
        'ic95': intervalles_bootstrap(y_test, p_test, seuil),
        'n_test': len(y_test), 'n_test_positifs': int(y_test.sum()),
        'poids_positif': float(poids_positif),
        'epoques_phase1': fin_phase1, 'epoques_total': len(historique), 'epoque_retenue': epoque_retenue,
        'duree_entrainement_min': duree / 60,
        'gradcam': sauver_gradcam(model, test, dossier),
    }
    (dossier / 'metriques.json').write_text(json.dumps(RESULTATS[cle], indent=2, ensure_ascii=False),
                                             encoding='utf-8')
    m = RESULTATS[cle]['test']
    print(f"→ test : AUC {m['auc']:.3f} | rappel {m['rappel']:.3f} | spécificité {m['specificite']:.3f} "
          f"| F1 {m['f1']:.3f} | seuil {seuil:.3f} | {duree / 60:.0f} min")
    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

## 9. Tableau 3.1 : métriques par pathologie (ensemble de test)

In [ ]:
def fr(x, d=3):
    return f'{x:.{d}f}'.replace('.', ',')


def avec_ic(r, k):
    bas, haut = r['ic95'][k]
    return f"{fr(r['test'][k])} [{fr(bas)} ; {fr(haut)}]"


METRIQUES = [('precision', 'Précision'), ('rappel', 'Rappel (sensibilité)'), ('specificite', 'Spécificité'),
             ('f1', 'F1-score'), ('auc', 'AUC')]
lignes = []
for cle in TACHES_A_LANCER:
    r = RESULTATS[cle]
    lignes.append({'Tâche': r['titre'], **{lib: avec_ic(r, k) for k, lib in METRIQUES},
                   'Images de test': r['n_test']})
pathos = [RESULTATS[c] for c in TACHES_A_LANCER if RESULTATS[c]['pathologie']]
if len(pathos) > 1:
    lignes.append({'Tâche': 'Moyenne (pathologies)',
                   **{lib: fr(np.mean([r['test'][k] for r in pathos])) for k, lib in METRIQUES},
                   'Images de test': sum(r['n_test'] for r in pathos)})
lignes.append({'Tâche': 'Référence : CheXNet (Rajpurkar et al., 2017)', 'Précision': '',
               'Rappel (sensibilité)': '', 'Spécificité': '', 'F1-score': '0,435', 'AUC': '',
               'Images de test': ''})
TABLEAU_3_1 = pd.DataFrame(lignes)
TABLEAU_3_1.to_csv(OUT_DIR / 'tableau3_1.csv', index=False, encoding='utf-8-sig')
TABLEAU_3_1

## 10. Figures 3.1 à 3.4

In [ ]:
fig, axes = plt.subplots(len(TACHES_A_LANCER), 2, figsize=(12, 3.6 * len(TACHES_A_LANCER)), squeeze=False)
for ligne, cle in enumerate(TACHES_A_LANCER):
    h, r = pd.read_csv(OUT_DIR / cle / 'historique.csv'), RESULTATS[cle]
    for col, (k, nom) in enumerate([('loss', 'Perte (entropie croisée binaire)'),
                                     ('acc', 'Exactitude (accuracy)')]):
        ax = axes[ligne][col]
        ax.plot(h.epoque, h[k], color='#2a6fb0', lw=2, label='Entraînement')
        ax.plot(h.epoque, h[f'val_{k}'], color='#d9822b', lw=2, ls='--', label='Validation')
        ax.axvline(r['epoques_phase1'] + 0.5, color='#4a9a4a', ls=':', lw=1.8, label='Début du fine-tuning')
        ax.axvline(r['epoque_retenue'], color='#c0392b', ls=':', lw=1.8, label='Époque retenue')
        ax.set_title(f"{r['titre']} : {nom}", fontsize=10)
        ax.set_xlabel('Époque')
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
        if ligne == 0 and col == 0:
            ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'figure3_1_courbes_apprentissage.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(TACHES_A_LANCER), figsize=(4.4 * len(TACHES_A_LANCER), 4), squeeze=False)
for ax, cle in zip(axes[0], TACHES_A_LANCER):
    r = RESULTATS[cle]
    m = r['test']
    mc = np.array([[m['vn'], m['fp']], [m['fn'], m['vp']]])
    ax.imshow(mc, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{mc[i, j]}\n({mc[i, j] / mc[i].sum():.0%})".replace('.', ','), ha='center',
                    va='center', fontsize=11, color='white' if mc[i, j] > mc.max() / 2 else 'black')
    ax.set_xticks([0, 1], r['libelles'])
    ax.set_yticks([0, 1], r['libelles'])
    ax.set_xlabel('Classe prédite')
    ax.set_ylabel('Classe réelle')
    ax.set_title(r['titre'], fontsize=10)
    ax.spines[:].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'figure3_2_matrices_confusion.png', bbox_inches='tight')
plt.show()

In [ ]:
couleurs = ['#2a6fb0', '#c0392b', '#4a9a4a', '#8e44ad']
fig, ax = plt.subplots(figsize=(6, 6))
for c, cle in zip(couleurs, TACHES_A_LANCER):
    r = RESULTATS[cle]
    pred = pd.read_csv(OUT_DIR / cle / 'predictions_test.csv')
    fpr, tpr, _ = roc_curve(pred.y, pred.proba)
    bas, haut = r['ic95']['auc']
    ax.plot(fpr, tpr, color=c, lw=2,
            label=f"{r['titre']} : AUC = {fr(r['test']['auc'])} (IC 95 % : {fr(bas)} ; {fr(haut)})")
    ax.plot(1 - r['test']['specificite'], r['test']['rappel'], 'o', color=c, ms=7)
ax.plot([0, 1], [0, 1], color='grey', ls='--', lw=1, label='Classement aléatoire (AUC = 0,5)')
ax.set_xlabel('Taux de faux positifs (1 − spécificité)')
ax.set_ylabel('Sensibilité (rappel)')
ax.set_title("Courbes ROC sur l'ensemble de test\n(● : seuil de décision retenu en validation)", fontsize=10)
ax.legend(fontsize=7.5, loc='lower right')
ax.grid(alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(OUT_DIR / 'figure3_3_courbes_roc.png', bbox_inches='tight')
plt.show()

In [ ]:
taches_cam = [c for c in TACHES_A_LANCER if RESULTATS[c]['pathologie']]
fig, axes = plt.subplots(len(taches_cam), 2, figsize=(8, 4.2 * len(taches_cam)), squeeze=False)
for ligne, cle in enumerate(taches_cam):
    r = RESULTATS[cle]
    for col, (type_cas, nom) in enumerate([('VP', 'Vrai positif'), ('FN', 'Faux négatif')]):
        ax = axes[ligne][col]
        ax.axis('off')
        fichier = OUT_DIR / cle / f'gradcam_{type_cas}.png'
        if fichier.exists():
            ax.imshow(plt.imread(fichier))
            ax.set_title(f"{r['titre']} : {nom}\nscore = {fr(r['gradcam'][type_cas]['proba'], 2)}", fontsize=10)
        else:
            ax.set_title(f"{r['titre']} : {nom}\n(aucun cas dans le test)", fontsize=10)
cbar = fig.colorbar(plt.cm.ScalarMappable(cmap='jet'), ax=axes, fraction=0.03, pad=0.02)
cbar.set_label('Contribution relative à la décision (Grad-CAM)')
plt.savefig(OUT_DIR / 'figure3_4_gradcam.png', bbox_inches='tight')
plt.show()

## 11. Récapitulatif pour la rédaction et téléchargement

In [ ]:
def ligne_metriques(m):
    return (f"précision {fr(m['precision'])} · rappel {fr(m['rappel'])} · spécificité {fr(m['specificite'])} · "
            f"F1 {fr(m['f1'])} · AUC {fr(m['auc'])} (VP {m['vp']}, FP {m['fp']}, FN {m['fn']}, VN {m['vn']})")


texte = [
    '# Résultats expérimentaux : récapitulatif pour la rédaction', '',
    f"Exécuté le {datetime.date.today().isoformat()} · {GPU} · PyTorch {torch.__version__} · "
    f"torchvision {torchvision.__version__} · Python {platform.python_version()}", '',
    '## Configuration', '', '```json', json.dumps(CONFIG, indent=2, ensure_ascii=False), '```', '',
    '## Répartition des données (partitionnement par patient)', '', REPARTITION.to_markdown(index=False), '',
]
for cle in TACHES_A_LANCER:
    r = RESULTATS[cle]
    texte += [
        f"## {r['titre']}", '',
        f"- Entraînement : {r['epoques_phase1']} époques en phase 1, {r['epoques_total'] - r['epoques_phase1']} en "
        f"phase 2, modèle retenu à l'époque {r['epoque_retenue']} · {fr(r['duree_entrainement_min'], 0)} min",
        f"- Poids des positifs dans la perte : {fr(r['poids_positif'], 2)}",
        f"- Seuil choisi en validation (Youden) : {fr(r['seuil_validation'])}",
        f"- Test, seuil de validation : {ligne_metriques(r['test'])}",
        f"- Test, seuil 0,5 : {ligne_metriques(r['test_seuil_0_5'])}",
        '- IC 95 % (bootstrap, ' + str(CONFIG['bootstrap']) + ' tirages) : ' +
        ' · '.join(f"{k} {fr(b)} ; {fr(h)}" for k, (b, h) in r['ic95'].items()),
        f"- Grad-CAM : {json.dumps(r['gradcam'], ensure_ascii=False)}", '',
    ]
texte += ['## Tableau 3.1', '', TABLEAU_3_1.to_markdown(index=False), '']
(OUT_DIR / 'resume_pour_memoire.md').write_text('\n'.join(texte), encoding='utf-8')
print('\n'.join(texte))

# Les modèles (.pt, ~30 Mo chacun) restent sur le Drive : l'archive ne contient que les résultats
import zipfile
archive = Path('/content/memoire_resultats.zip' if IN_COLAB else 'memoire_resultats.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in OUT_DIR.rglob('*'):
        if f.is_file() and f.suffix != '.pt':
            z.write(f, f.relative_to(OUT_DIR))
if IN_COLAB:
    colab_files.download(archive)
print(f'Archive : {archive}')